# LiViFuser confirmatory-v3 simulation training (T4×2)

Attach exactly three Kaggle datasets before running: (1) the frozen training-code bundle, (2) all 18 files from `gpu_handoff_train_val_v1`, and (3) `livifuser_dinov3_splus_cache_v2_bundle.zip`. Do **not** attach the held-out handoff or held-out cache. Select the **GPU T4 ×2** accelerator and run all cells. The preparation and runner scripts fail closed on held-out inputs, contract drift, count drift, or a non-T4×2 runtime.

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path, PurePosixPath

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
candidates = list(INPUT.rglob('cloud_bundle_manifest.json'))
if not candidates:
    archives = list(INPUT.rglob('livifuser_sim_training_code_*.zip'))
    assert len(archives) == 1, f'expected one code archive, found {len(archives)}'
    code_extract = WORK / 'livifuser_code'
    assert not code_extract.exists(), f'refusing existing extraction root: {code_extract}'
    with zipfile.ZipFile(archives[0]) as archive:
        for info in archive.infolist():
            member = PurePosixPath(info.filename)
            assert not member.is_absolute() and '..' not in member.parts, f'unsafe code member: {info.filename}'
        archive.extractall(code_extract)
    candidates = list(code_extract.rglob('cloud_bundle_manifest.json'))
assert len(candidates) == 1, f'expected one code manifest, found {len(candidates)}'
REPO = candidates[0].parent
sys.path.insert(0, str(REPO / 'src'))
from livifuser_nav.cloud_bundle import verify_cloud_bundle

verification = verify_cloud_bundle(REPO)
config = REPO / 'config/simulation_sweep_v1.json'
config_sha = hashlib.sha256(config.read_bytes()).hexdigest().upper()
assert config_sha == '76680BCE45A67D5D91F42660D5EC25F90450B281838CE311E329477C4E36F09E'
compile_env = os.environ.copy()
compile_env['PYTHONPYCACHEPREFIX'] = str(WORK / 'livifuser_pycache')
subprocess.run([sys.executable, '-m', 'compileall', '-q', str(REPO / 'src'), str(REPO / 'scripts')], check=True, env=compile_env)
print(json.dumps({'repository': str(REPO), 'cloud_bundle': verification, 'config_sha256': config_sha}, indent=2))

In [ ]:
import torch

assert torch.cuda.is_available(), 'enable the Kaggle GPU T4 ×2 accelerator'
devices = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
assert len(devices) == 2 and all('T4' in name for name in devices), f'expected T4 ×2, found {devices}'
print(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda, 'devices': devices}, indent=2))

In [ ]:
DATA_ROOT = WORK / 'livifuser_sim_training_data_v1'
DATA_PLAN = WORK / 'livifuser_sim_training_data_plan_v1.json'
prepare_command = [
    sys.executable, str(REPO / 'scripts/prepare_sim_training_data.py'),
    '--input-root', str(INPUT),
    '--work-root', str(DATA_ROOT),
    '--plan-output', str(DATA_PLAN),
]
print(' '.join(prepare_command))
subprocess.run(prepare_command, cwd=REPO, check=True)
plan = json.loads(DATA_PLAN.read_text())
assert plan['train']['episode_count'] == 120 and plan['train']['windows_k8_h8'] == 41367
assert plan['validation']['episode_count'] == 30 and plan['validation']['windows_k8_h8'] == 9459
print(json.dumps({'plan': str(DATA_PLAN), 'train': plan['train']['episode_count'], 'validation': plan['validation']['episode_count']}, indent=2))

In [ ]:
RESULTS = WORK / 'livifuser_simulation_sweep_v1'
train_command = [
    sys.executable, str(REPO / 'scripts/run_simulation_sweep_kaggle.py'),
    '--data-plan', str(DATA_PLAN),
    '--config', str(config),
    '--output-root', str(RESULTS),
    '--cuda-device', '0', '--cuda-device', '1',
]
print(' '.join(train_command))
subprocess.run(train_command, cwd=REPO, check=True)
summary = json.loads((RESULTS / 'summary.json').read_text())
assert summary['result_count'] == 24 and summary['config_sha256'] == config_sha
print(json.dumps({'result_count': summary['result_count'], 'summary': str(RESULTS / 'summary.json')}, indent=2))

In [ ]:
import shutil

summary_path = RESULTS / 'summary.json'
result_manifest = {
    'schema_version': 1,
    'config_sha256': config_sha,
    'data_plan_sha256': hashlib.sha256(DATA_PLAN.read_bytes()).hexdigest().upper(),
    'summary_sha256': hashlib.sha256(summary_path.read_bytes()).hexdigest().upper(),
    'result_count': 24,
}
(RESULTS / 'RESULT_BUNDLE_MANIFEST.json').write_text(json.dumps(result_manifest, indent=2) + '\n')
archive_base = WORK / 'livifuser_simulation_sweep_v1_results'
archive_path = archive_base.with_suffix('.zip')
assert not archive_path.exists(), f'refusing to overwrite {archive_path}'
shutil.make_archive(str(archive_base), 'zip', root_dir=RESULTS.parent, base_dir=RESULTS.name)
digest = hashlib.sha256()
with archive_path.open('rb') as handle:
    while chunk := handle.read(8 * 1024 * 1024):
        digest.update(chunk)
archive_sha = digest.hexdigest().upper()
print(json.dumps({'download': str(archive_path), 'size_bytes': archive_path.stat().st_size, 'sha256': archive_sha, **result_manifest}, indent=2))